In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime
import warnings
import kagglehub

## 1. Project Overview

Hypothesis Summary: Weather and Tourist Arrivals in Singapore

**Research question:** To what extent do weather conditions in Singapore explain changes in tourist arrivals, and how can the Singapore Tourism Board (STB) use this analysis to increase tourism?

### Hypotheses

| | Statement |
|---|---|
| **H₀ (null)** | Weather conditions (rainfall, rainy days, temperature, sunshine, humidity) have **no significant effect** on monthly tourist arrivals to Singapore. |
| **H₁ (working hypothesis)** | Weather conditions, such as the monsoon seasons, **significantly influence** monthly tourist arrival volumes. |

**Type of Analysis**: This project will utilize Time-Series Regression Analysis. While Singapore possesses a relatively static tropical climate year-round, granular weather data (e.g., monthly rainfall, humidity, and temperature variations) exhibits certain level of variance to impact travel decisions. By analyzing historical weather and arrival data, we aim to build a model that predicts arrival peaks and troughs.

**Targeted Quality Metric**: The performance of our predictive model will be evaluated using Mean Absolute Error (MAE) to understand the average deviation in our predicted visitor counts, alongside R-squared ($R^2$) to determine the proportion of variance in tourist arrivals that can be explained by our weather variables.

### Data Used
- SingStat monthly weather data (rainfall, rainy days, max temperature, sunshine hours, humidity) 
- SingStat monthly tourist arrivals, 2008–2025
- The Kaggle Global Weather Repository, used as secondary source to validate the SingStat monthly weather data.



## 2.Data Source Identification and EDA


*   **Singapore Weather Data**
*   [Global Weather Repository (DS1)](https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository/data/discussion)
*   [Singapore Air Temperature, Sunshine, Relative Humidity and Rainfall in Singapore (DS2)](https://tablebuilder.singstat.gov.sg/table/TS/M890081)
*   **Singapore Tourism Data**
*   [Singapore Tourist Arrival (DS3)](https://tablebuilder.singstat.gov.sg/table/TS/M5550001)

In [ ]:
import kagglehub
import glob
import os
import shutil
import pandas as pd
import numpy as np

# Save the dataset in a 'data' folder next to this notebook (inside the repository)
DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Only download from Kaggle if the CSV is not already in the repository
if not glob.glob(os.path.join(DATA_DIR, "*.csv")):
    # Download latest version of the dataset from Kaggle (saved to the kagglehub cache first)
    path = kagglehub.dataset_download("nelgiriyewithana/global-weather-repository")
    print("Downloaded to Kaggle cache:", path)

    # Copy the CSV file(s) from the cache into the repository's data folder
    for f in glob.glob(os.path.join(path, "*.csv")):
        shutil.copy(f, DATA_DIR)

print("Path to dataset files:", DATA_DIR)

# Load the CSV from the repository's data folder
file_path = glob.glob(os.path.join(DATA_DIR, "*.csv"))[0]
df = pd.read_csv(file_path)

##### 2.3.1.1 Data Profiling for Source 1

In [ ]:
# Data is loaded from Kaggle in the cell above (df)
country_col = df.columns[0]

#We only want data for Singapore
singapore_data = df[df[country_col].astype(str).str.strip() == 'Singapore']
singapore_data.to_csv('singapore_only.csv', index=False)

print("Data Profiling for Source 1: Global Weather Repository")
print("="*60)

df = singapore_data  # your existing Source 1 dataframe

print("\n1. STRUCTURE INSPECTION")
print("-" * 60)
print(f"Source 1 has {df.shape[0]} rows and {df.shape[1]} columns")

print("\n2. DATA TYPES & MISSING VALUES")
print("-" * 60)
print("\nColumn Data Types:")
print(df.dtypes)
print("\nMissing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_summary[missing_summary['Missing Count'] > 0])

print("\n3. STATISTICAL PROFILING (Numeric Columns)")
print("-" * 60)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns found: {len(numeric_cols)}")
print(numeric_cols[:10])
if len(numeric_cols) > 0:
    print("\nSummary Statistics:")
    print(df[numeric_cols].describe())

print("\n4. CATEGORICAL COLUMNS ANALYSIS")
print("-" * 60)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns found: {len(categorical_cols)}")
for col in categorical_cols[:5]:
    print(f"\n'{col}' - Unique values: {df[col].nunique()}")
    if df[col].nunique() <= 20:
        print(df[col].value_counts())
    else:
        print(f"Top 10 values:")
        print(df[col].value_counts().head(10))

print("\n5. DATE/DATETIME ANALYSIS")
print("-" * 60)
date_cols = df.select_dtypes(include=['datetime64']).columns.tolist()
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        if col not in date_cols:
            try:
                df[col + '_converted'] = pd.to_datetime(df[col], errors='coerce')
                if df[col + '_converted'].notna().sum() > len(df) * 0.5:
                    date_cols.append(col)
                    print(f"Found potential date column: '{col}'")
            except:
                pass
if len(date_cols) > 0:
    for col in date_cols:
        if col in df.columns:
            print(f"\n'{col}':")
            try:
                date_range = df[col].max() - df[col].min()
                print(f"  Date range: {df[col].min()} to {df[col].max()}")
                if hasattr(date_range, 'days'):
                    print(f"  Total days: {date_range.days}")
            except:
                print(f"  Range: {df[col].min()} to {df[col].max()}")
else:
    print("No date columns detected.")

print("\n6. DATA QUALITY CHECKS")
print("-" * 60)
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
if len(numeric_cols) > 0:
    print("\nZero values in numeric columns:")
    for col in numeric_cols[:10]:
        zero_count = (df[col] == 0).sum()
        if zero_count > 0:
            print(f"  {col}: {zero_count} zeros ({zero_count/len(df)*100:.2f}%)")

print("\n7. CORRELATION ANALYSIS (Top Numeric Columns)")
print("-" * 60)
if len(numeric_cols) >= 2:
    corr_cols = numeric_cols[:min(10, len(numeric_cols))]
    corr_matrix = df[corr_cols].corr()
    strong_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) > 0.7:
                strong_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_val))
    if strong_corr:
        print("\nStrong correlations found (|r| > 0.7):")
        for col1, col2, val in strong_corr[:10]:
            print(f"  {col1} <-> {col2}: {val:.3f}")
    else:
        print("No strong correlations found (|r| > 0.7)")
else:
    print("Insufficient numeric columns for correlation analysis.")

print("\n8. UNIQUE IDENTIFIERS CHECK")
print("-" * 60)
for col in df.columns[:10]:
    unique_ratio = df[col].nunique() / len(df)
    if unique_ratio == 1.0:
        print(f"'{col}' - Potential unique identifier (100% unique)")
    elif unique_ratio > 0.95:
        print(f"'{col}' - High uniqueness ({unique_ratio*100:.1f}%)")

print("\n9. DATA QUALITY DISCUSSION (Balanced Evaluation)")
print("-" * 60)

overall_missing_pct = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100

print("\nCompleteness:")
if overall_missing_pct < 5:
    print(f"  (+) The dataset is highly complete with only {overall_missing_pct:.2f}% missing values, reducing the need for extensive data cleaning.")
else:
    print(f"  (-) The dataset has {overall_missing_pct:.2f}% missing values, which may require imputation or removal of incomplete records during preprocessing.")

print("\nReliability:")
print(f"  (-) Data is downloaded from \"Kaggle.\" and hence might not be as reliable as those other data from governmental sources. Cross referencing is required.")

print("\nTimeliness:")
if len(date_cols) > 0:
    try:
        # Convert Unix timestamp to datetime
        min_ts = df[date_cols[0]].min()
        max_ts = df[date_cols[0]].max()

        min_date = pd.to_datetime(min_ts, unit='s')
        max_date = pd.to_datetime(max_ts, unit='s')

        print(f"  (+) The dataset is updated daily from {min_date} to {max_date}, allowing for temporal analysis.")
    except:
        print("  (-) Date columns exist but could not be converted properly.")
else:
    print("  (-) No date information available.")


##### 2.3.1.2 Data Structuring for Source 1

In [ ]:
print("Data Structuring for Source 1: Global Weather Repository")
print("="*60)

# Filter to Singapore only
df_s1 = df[df['country'].str.lower() == 'singapore'].copy()
print(f"\nRows after Singapore filter: {len(df_s1)}")

# Parse datetime column
df_s1['last_updated'] = pd.to_datetime(df_s1['last_updated'], errors='coerce')
invalid_dates = df_s1['last_updated'].isna().sum()
print(f"Invalid datetime entries removed: {invalid_dates}")
df_s1 = df_s1[df_s1['last_updated'].notna()].copy()

# Extract month period for aggregation
df_s1['month'] = df_s1['last_updated'].dt.to_period('M')

# Keep only relevant columns for analysis
s1_keep = ['month', 'temperature_celsius', 'humidity', 'precip_mm']
df_s1 = df_s1[s1_keep].copy()

# Standardise column names
df_s1.rename(columns={
    'temperature_celsius' : 'temp_celsius',
    'humidity'            : 'humidity_pct',
    'precip_mm'           : 'rainfall_mm'
}, inplace=True)

print(f"\nColumns retained : {df_s1.columns.tolist()}")
print(f"Shape            : {df_s1.shape}")
print(f"Date range       : {df_s1['month'].min()} to {df_s1['month'].max()}")
print("\nFirst 5 rows:")
print(df_s1.head())

##### 2.3.1.3 Data Enriching for Source 1

In [ ]:
print("Data Enriching for Source 1: Global Weather Repository")
print("="*60)

# Aggregate daily readings to monthly, since Source 2 and 3 are both in monthly granaluarity
print("\nAggregating daily to monthly...")
df_s1_monthly = df_s1.groupby('month').agg(
    temp_celsius = ('temp_celsius', 'mean'),
    humidity_pct = ('humidity_pct', 'mean'),
    rainfall_mm  = ('rainfall_mm',  'sum')   # total monthly rainfall
).reset_index()

# Derive month number and year
df_s1_monthly['year']      = df_s1_monthly['month'].dt.year
df_s1_monthly['month_num'] = df_s1_monthly['month'].dt.month

# Add Singapore meteorological season
def get_season(m):
    if m in [12, 1, 2]:        return 'Northeast Monsoon'
    elif m in [3, 4]:           return 'Inter-monsoon 1'
    elif m in [5, 6, 7, 8, 9]: return 'Southwest Monsoon'
    else:                       return 'Inter-monsoon 2'

df_s1_monthly['season'] = df_s1_monthly['month_num'].apply(get_season)

# 3-month rolling averages for key variables
df_s1_monthly['temp_3m_avg']     = df_s1_monthly['temp_celsius'].rolling(window=3, min_periods=1).mean().round(2)
df_s1_monthly['humidity_3m_avg'] = df_s1_monthly['humidity_pct'].rolling(window=3, min_periods=1).mean().round(2)
df_s1_monthly['rainfall_3m_avg'] = df_s1_monthly['rainfall_mm'].rolling(window=3, min_periods=1).mean().round(2)

print(f"Shape after aggregation : {df_s1_monthly.shape}")
print(f"New columns added       : year, month_num, season, temp_3m_avg, humidity_3m_avg, rainfall_3m_avg")
print("\nFirst 5 rows:")
print(df_s1_monthly.head())

##### 2.3.1.4 Data Cleaning for Source 1

In [ ]:
print("Data Cleaning for Source 1: Global Weather Repository")
print("="*60)

# Remove duplicate months
before = len(df_s1_monthly)
df_s1_monthly = df_s1_monthly.drop_duplicates(subset='month').sort_values('month').reset_index(drop=True)
print(f"\nDuplicate months removed : {before - len(df_s1_monthly)}")

# Validate realistic ranges for Singapore weather
print("\nApplying range validation...")
invalid_temp     = (~df_s1_monthly['temp_celsius'].between(20, 40)).sum()
invalid_humidity = (~df_s1_monthly['humidity_pct'].between(0, 100)).sum()
invalid_rainfall = (df_s1_monthly['rainfall_mm'] < 0).sum()

df_s1_monthly.loc[~df_s1_monthly['temp_celsius'].between(20, 40),  'temp_celsius'] = np.nan
df_s1_monthly.loc[~df_s1_monthly['humidity_pct'].between(0, 100),  'humidity_pct'] = np.nan
df_s1_monthly.loc[df_s1_monthly['rainfall_mm'] < 0,                'rainfall_mm']  = np.nan

print(f"  temp_celsius  : {invalid_temp} out-of-range values replaced with NaN")
print(f"  humidity_pct  : {invalid_humidity} out-of-range values replaced with NaN")
print(f"  rainfall_mm   : {invalid_rainfall} negative values replaced with NaN")

# Interpolate missing values (if there even is any)
print("\nInterpolating missing values...")
df_s1_monthly[['temp_celsius', 'humidity_pct', 'rainfall_mm']] = (
    df_s1_monthly[['temp_celsius', 'humidity_pct', 'rainfall_mm']]
    .interpolate(method='linear', limit_direction='both')
)

print(f"Remaining nulls : {df_s1_monthly.isnull().sum().sum()}")
print(f"Final shape     : {df_s1_monthly.shape}")
print("\nFirst 5 rows:")
print(df_s1_monthly.head())

##### 2.3.1.5 Potential areas to conduct further analysis

1. **Time Series Analysis:** Examine trends over the 622-day period
2. **Correlation Studies:** Investigate relationships between variables (e.g., temperature vs. humidity)
3. **Seasonal Decomposition:** Identify cyclical patterns
4. **Extreme Event Prediction:** Model conditions leading to heavy rain or high UV
5. **Air Quality Modeling:** Analyze factors contributing to PM2.5 spikes
6. **Comparison:** Compare with historical Singapore climate data


